# EP2 - Membros FINAIS do ensemble (Colab)Treina as sementes do BERT em **TODO** o train e gera as probabilidades de`test1` e `test2`. Isso é o passo de entrega: o 0,4760 foi medido no holdout com3 sementes treinadas em 80% dos dados; aqui refitamos em 100%, que é a práticapadrão (medir no holdout, refitar em tudo).Hiperparâmetros idênticos aos membros medidos: lr 2e-5, 4 épocas, batch 8(aqui batch 16 com lr 2e-5 para aproveitar a GPU; mesma ordem de grandeza).**Saídas** (baixadas no fim, me mande todas):- `probs_final_test1_seed<N>.npz`- `probs_final_test2_seed<N>.npz`**Antes de rodar:** `Ambiente de execução > Alterar o tipo > GPU` (L4 de preferência).Tempo: ~40 min por semente na L4; o grid abaixo roda 3 sementes (~2h).**Faça upload de 3 arquivos:** `train.xlsx`, `test1.xlsx`, `test2.xlsx`.

In [ ]:
!nvidia-smi
!pip install -q transformers accelerate openpyxl

In [ ]:
from google.colab import files
up = files.upload()   # selecione train.xlsx, test1.xlsx e test2.xlsx (pode marcar os 3)

In [ ]:
import numpy as np, pandas as pd, torch, os
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, EarlyStoppingCallback,
                          Trainer, TrainingArguments)

MODEL = "neuralmind/bert-base-portuguese-cased"
LABELS = ["c1", "c234", "c5"]
LABEL_MAP = {l: i for i, l in enumerate(LABELS)}
MAX_LEN = 512
SEEDS = [42, 1, 2]        # as 3 sementes do ensemble final

train = pd.read_excel("train.xlsx").dropna(subset=["resp_text", "clarity"])
train["resp_text"] = train["resp_text"].astype(str)
train["clarity"] = train["clarity"].astype(str).str.strip()
train = train[train["clarity"].isin(LABELS)].copy()
train["label"] = train["clarity"].map(LABEL_MAP)
print(f"train completo: {len(train)}")

tests = {}
for name in ["test1", "test2"]:
    t = pd.read_excel(f"{name}.xlsx")
    t["resp_text"] = t["resp_text"].fillna("").astype(str)
    tests[name] = t
    print(f"{name}: {len(t)} linhas")

tokenizer = AutoTokenizer.from_pretrained(MODEL)
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"GPU: {torch.cuda.get_device_name(0)} | bf16={USE_BF16}")

class DS(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = list(labels) if labels is not None else None
    def __getitem__(self, i):
        d = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None: d["labels"] = self.labels[i]
        return d
    def __len__(self): return len(self.enc["input_ids"])

def compute_metrics(p):
    return {"f1_macro": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")}

## Treino das 3 sementesSalva as probabilidades **a cada semente**, então se a sessão cair você só perdea semente em andamento. Pode rodar a célula de novo: ela pula as já prontas.

In [ ]:
def run_seed(seed):
    # TODO o train; 10% apenas para early stopping
    tr, va = train_test_split(train, test_size=0.1, stratify=train["label"],
                              random_state=seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
    args = TrainingArguments(
        output_dir=f"./out_f{seed}",
        num_train_epochs=4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        learning_rate=2e-5,
        warmup_steps=500,
        weight_decay=0.01,
        fp16=not USE_BF16, bf16=USE_BF16, max_grad_norm=1.0,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="f1_macro",
        greater_is_better=True, save_total_limit=1,
        seed=seed, report_to="none", logging_steps=200,
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=DS(tr["resp_text"], tr["label"]),
                      eval_dataset=DS(va["resp_text"], va["label"]),
                      compute_metrics=compute_metrics,
                      data_collator=DataCollatorWithPadding(tokenizer),
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer.train()
    for name, t in tests.items():
        pr = trainer.predict(DS(t["resp_text"]))
        probs = torch.softmax(torch.tensor(pr.predictions.astype("float32")), -1).numpy()
        np.savez(f"probs_final_{name}_seed{seed}.npz", probs=probs)
        d = pd.Series([LABELS[i] for i in probs.argmax(1)]).value_counts(normalize=True)
        print(f"  {name} -> probs_final_{name}_seed{seed}.npz | {d.reindex(LABELS).round(3).to_dict()}")
    del model, trainer
    torch.cuda.empty_cache()

for s in SEEDS:
    if os.path.exists(f"probs_final_test2_seed{s}.npz"):
        print(f"semente {s}: ja pronta, pulando")
        continue
    print(f"\n===== semente {s} =====")
    run_seed(s)
print("\nTODAS AS SEMENTES PRONTAS")

## Download (me mande todos os arquivos)

In [ ]:
for s in SEEDS:
    for name in ["test1", "test2"]:
        f = f"probs_final_{name}_seed{s}.npz"
        if os.path.exists(f):
            print("baixando", f)
            files.download(f)